In [1]:
#IMPORT LIBRARIES
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#LOAD DATASET

df = pd.read_csv("AccidentsBig.csv")

#DROP USELESS COLUMNS

drop_cols = [
    "Accident_Index",
    "Police_Force",
    "Local_Authority_(District)",
    "Local_Authority_(Highway)",
    "1st_Road_Number",
    "2nd_Road_Number",
    "LSOA_of_Accident_Location",
    "Did_Police_Officer_Attend_Scene_of_Accident"
]

df.drop(columns=drop_cols, inplace=True, errors="ignore")


df["Time"] = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce")
df["Time_numeric"] = df["Time"].dt.hour + df["Time"].dt.minute / 60
df.drop(columns=["Time"], inplace=True)


df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30", errors="coerce")

df["Month"] = df["Date"].dt.month
df["Is_Weekend"] = df["Date"].dt.weekday >= 5

df.drop(columns=["Date"], inplace=True)


df.replace(-1, np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)


severity_map = {1:2, 2:1, 3:0}
df["Accident_Severity"] = df["Accident_Severity"].apply(
    lambda x:0 if x==3 else 1
)


X = df.drop(columns=["Accident_Severity"])
y = df["Accident_Severity"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

probabilities = model.predict_proba(X_test)
high_risk_prob = probabilities[:, 1]

threshold = 0.35
custom_pred = (high_risk_prob > threshold).astype(int)

print("\nAccuracy:", accuracy_score(y_test, custom_pred))
print("\nClassification Report:\n", classification_report(y_test, custom_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, custom_pred))


importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\nTop Important Features:\n")
print(importance.head(10))


joblib.dump(model, "accident_rf_model.pkl")
print("\nModel saved successfully!")



Accuracy: 0.7406666666666667

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.79      0.84     10395
           1       0.24      0.43      0.31      1605

    accuracy                           0.74     12000
   macro avg       0.57      0.61      0.57     12000
weighted avg       0.81      0.74      0.77     12000


Confusion Matrix:
 [[8193 2202]
 [ 910  695]]

Top Important Features:

longitude               0.150630
latitude                0.149754
Time_numeric            0.140996
Number_of_Vehicles      0.083174
Month                   0.078574
Day_of_Week             0.056442
Junction_Detail         0.044592
1st_Road_Class          0.037786
Number_of_Casualties    0.037094
Road_Type               0.027778
dtype: float64

Model saved successfully!


In [ ]:
print(X.columns)

Index(['longitude', 'latitude', 'Number_of_Vehicles', 'Number_of_Casualties',
       'Day_of_Week', '1st_Road_Class', 'Road_Type', 'Speed_limit',
       'Junction_Detail', 'Junction_Control', '2nd_Road_Class',
       'Pedestrian_Crossing-Human_Control',
       'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions',
       'Weather_Conditions', 'Road_Surface_Conditions',
       'Special_Conditions_at_Site', 'Carriageway_Hazards',
       'Urban_or_Rural_Area', 'Time_numeric', 'Month', 'Is_Weekend'],
      dtype='object')


In [ ]:
X = df.drop(columns=["Accident_Severity"])

